## OpenAI Agents SDK 

In [ ]:
%%bash
uv pip install openai-agents

## import env

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
if not os.getenv("OPENAI_BASE_URL"):
   raise ValueError("OPENAI_BASE_URL not found in .env file. Please add it.")
if not os.getenv("OPENAI_API_KEY"):
   raise ValueError("OPENAI_API_KEY not found in .env file. Please add it.")

## run first agents

In [33]:
from agents import ToolCallItem, ToolCallOutputItem

def print_usage(result):
    usage = result.context_wrapper.usage
    input_details = usage.input_tokens_details
    output_details = usage.output_tokens_details

    print("=" * 60)
    print("Usage Summary")
    print("=" * 60)
    print(f"Requests         : {usage.requests}")
    print(f"Input Tokens     : {usage.input_tokens}")
    print(f"Output Tokens    : {usage.output_tokens}")
    print(f"Total Tokens     : {usage.total_tokens}")
    print(f"Cached Tokens    : {input_details.cached_tokens}")
    print(f"Cache Write      : {input_details.cache_write_tokens}")
    print(f"Reasoning Tokens : {output_details.reasoning_tokens}")

    print("\nLLM Requests")
    print("-" * 60)
    for i, req in enumerate(usage.request_usage_entries, 1):
        print(
            f"{i:2d}. "
            f"in        = {req.input_tokens:<5} "
            f"out       = {req.output_tokens:<5} "
            f"reasoning = {req.output_tokens_details.reasoning_tokens:<5} "
            f"cache     = {req.input_tokens_details.cached_tokens:<5} "
            f"total     = {req.total_tokens}"
        )
    print("\nTool Calls")
    print("-" * 60)

    found = False
    for item in result.new_items:

        if isinstance(item, ToolCallItem):
            found = True
            print(f"🔧 {getattr(item.raw_item, 'name', '<unknown>')}")

        elif isinstance(item, ToolCallOutputItem):
            print(f"   ↳ Output: {str(item.output)[:120]}")
    if not found:
        print("No tool calls.")

In [ ]:
from agents import Agent, ModelSettings, Runner
from openai.types.shared.reasoning import Reasoning

agent = Agent(
    name="History Tutor",
    model="gpt-5.6-luna",
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        include_usage=True,
    ),

    instructions="You answer history questions clearly and concisely.",
)

result = await Runner.run(agent, "When did the Roman Empire fall?")
print(result.final_output)
print_usage(result)

The **Western Roman Empire traditionally fell in 476 CE**, when the Germanic king **Odoacer deposed Emperor Romulus Augustulus**.

However, the **Eastern Roman Empire**, often called the Byzantine Empire, continued until **1453**, when Constantinople fell to the Ottoman Turks.
********** Token Usage **********
Requests      : 1
Total Tokens  : 90
Input Tokens  : 27
Output Tokens : 63
*********************************


## agent tools

In [34]:
from agents import Agent, Runner, function_tool

@function_tool
def history_fun_fact() -> str:
    return "Sharks are older than trees."

agent = Agent(
    name="History Tutor",
    instructions="Answer history questions clearly. Use history_fun_fact when it helps.",
    tools=[history_fun_fact],
)

result = await Runner.run(
    agent,
    "Tell me something surprising about ancient life on Earth.",
)
print(result.final_output)
print_usage(result)

Sharks are older than trees.
Usage Summary
Requests         : 2
Input Tokens     : 148
Output Tokens    : 26
Total Tokens     : 174
Cached Tokens    : 0
Cache Write      : 0
Reasoning Tokens : 0

LLM Requests
------------------------------------------------------------
 1. in        = 57    out       = 15    reasoning = 0     cache     = 0     total     = 72
 2. in        = 91    out       = 11    reasoning = 0     cache     = 0     total     = 102

Tool Calls
------------------------------------------------------------
🔧 history_fun_fact
   ↳ Output: Sharks are older than trees.


## agents as tools

In [35]:
from agents import Agent, Runner

history_tutor_agent = Agent(
    name="History Tutor",
    handoff_description="Specialist agent for historical questions",
    instructions="You answer history questions clearly and concisely.",
)

math_tutor_agent = Agent(
    name="Math Tutor",
    handoff_description="Specialist agent for math questions",
    instructions="You explain math step by step and include worked examples.",
)

triage_agent = Agent(
    name="Triage Agent",
    instructions="Route each homework question to the right specialist.",
    handoffs=[history_tutor_agent, math_tutor_agent],
)

result = await Runner.run(
        triage_agent,
        "Who was the first president of the United States?",
    )
print(result.final_output)
print(f"Answered by: {result.last_agent.name}")
print_usage(result)


George Washington was the first president of the United States.
Answered by: History Tutor
Usage Summary
Requests         : 2
Input Tokens     : 171
Output Tokens    : 32
Total Tokens     : 203
Cached Tokens    : 0
Cache Write      : 0
Reasoning Tokens : 0

LLM Requests
------------------------------------------------------------
 1. in        = 103   out       = 17    reasoning = 0     cache     = 0     total     = 120
 2. in        = 68    out       = 15    reasoning = 0     cache     = 0     total     = 83

Tool Calls
------------------------------------------------------------
No tool calls.
